In [4]:
# ============================================================
# CÉLULA 1 — CARREGAMENTO DOS DADOS
# ============================================================

import pandas as pd
import numpy as np

# Escolha a quantidade de partidas: 100, 1000 ou 10000
quantidade_partidas = 100

caminho_arquivo = f"../data/team-time-series-{quantidade_partidas}.xlsx"

df_series_temporais = pd.read_excel(caminho_arquivo)

display(df_series_temporais.head())

print(f"Arquivo carregado: {caminho_arquivo}")
print(f"Quantidade de linhas: {df_series_temporais.shape[0]}")
print(f"Quantidade de colunas: {df_series_temporais.shape[1]}")

print("\nColunas disponíveis:")
print(df_series_temporais.columns.tolist())

,match_id,timestep,teamId,vencedor,championStats.abilityHaste,championStats.abilityPower,championStats.armor,championStats.attackDamage,championStats.health,currentGold,...,diff_kills,diff_deaths,diff_assists,diff_dragons,diff_elder_dragons,diff_barons,diff_rift_heralds,diff_atakhan,diff_towers_destroyed,diff_inhibitors_destroyed
0,BR1_3288893506,0,100,100,0,0,162,303,3136,2100,...,0,0,0,0,0,0,0,0,0,0
1,BR1_3288893506,0,200,100,0,0,165,310,3252,2100,...,0,0,0,0,0,0,0,0,0,0
2,BR1_3288893506,1,100,100,0,75,162,331,3333,60,...,0,0,0,0,0,0,0,0,0,0
3,BR1_3288893506,1,200,100,0,57,173,333,3271,55,...,0,0,0,0,0,0,0,0,0,0
4,BR1_3288893506,2,100,100,0,72,199,344,1723,2321,...,2,-2,1,0,0,0,0,0,0,0


Arquivo carregado: ../data/team-time-series-100.xlsx
Quantidade de linhas: 5770
Quantidade de colunas: 74

Colunas disponíveis:
['match_id', 'timestep', 'teamId', 'vencedor', 'championStats.abilityHaste', 'championStats.abilityPower', 'championStats.armor', 'championStats.attackDamage', 'championStats.health', 'currentGold', 'damageStats.magicDamageDone', 'damageStats.magicDamageDoneToChampions', 'damageStats.magicDamageTaken', 'damageStats.physicalDamageDone', 'damageStats.physicalDamageDoneToChampions', 'damageStats.physicalDamageTaken', 'damageStats.totalDamageDone', 'damageStats.totalDamageDoneToChampions', 'damageStats.totalDamageTaken', 'damageStats.trueDamageDone', 'damageStats.trueDamageDoneToChampions', 'damageStats.trueDamageTaken', 'goldPerSecond', 'jungleMinionsKilled', 'level', 'minionsKilled', 'timeEnemySpentControlled', 'totalGold', 'xp', 'kills', 'deaths', 'assists', 'dragons', 'elder_dragons', 'barons', 'rift_heralds', 'atakhan', 'towers_destroyed', 'inhibitors_destroy

In [5]:
# ============================================================
# CÉLULA 2 — ORGANIZAÇÃO DAS SÉRIES TEMPORAIS
# ============================================================

# Colunas que não serão utilizadas como características de entrada
colunas_excluidas = [
    "match_id",
    "timestep",
    "teamId",
    "vencedor"
]

colunas_features = [
    coluna
    for coluna in df_series_temporais.columns
    if coluna not in colunas_excluidas
]

# Dicionários para armazenar as matrizes e os respectivos alvos
partidas = {}
vencedores = {}

# Organizar as linhas por partida, instante e equipe
dados_ordenados = df_series_temporais.sort_values(
    ["match_id", "timestep", "teamId"]
)

for match_id, grupo in dados_ordenados.groupby("match_id", sort=False):

    # Identificar o vencedor da partida
    valores_vencedor = grupo["vencedor"].dropna().unique()

    if len(valores_vencedor) != 1:
        continue

    vencedor = valores_vencedor[0]

    if vencedor not in (100, 200):
        continue

    sequencia = []
    partida_valida = True

    for timestep, instante in grupo.groupby("timestep", sort=True):

        # Cada instante deve conter exatamente uma linha por equipe
        equipes_presentes = instante["teamId"].tolist()

        if (
            len(equipes_presentes) != 2
            or sorted(equipes_presentes) != [100, 200]
        ):
            partida_valida = False
            break

        # Garantir a ordem Blue (100) e Red (200)
        instante_ordenado = (
            instante
            .set_index("teamId")
            .reindex([100, 200])
        )

        # Matriz do instante: (2 equipes, características)
        matriz_instante = instante_ordenado[
            colunas_features
        ].to_numpy(dtype=np.float32)

        sequencia.append(matriz_instante)

    if partida_valida and sequencia:
        # Matriz da partida: (instantes, 2 equipes, características)
        partidas[match_id] = np.stack(sequencia)
        vencedores[match_id] = int(vencedor)

# Resumo da organização
print(f"Partidas carregadas: {df_series_temporais['match_id'].nunique()}")
print(f"Partidas organizadas: {len(partidas)}")
print(f"Partidas descartadas na organização: "
      f"{df_series_temporais['match_id'].nunique() - len(partidas)}")
print(f"Características por equipe: {len(colunas_features)}")

if partidas:
    primeiro_match_id = next(iter(partidas))

    print(f"\nPartida de exemplo: {primeiro_match_id}")
    print(f"Formato da matriz: {partidas[primeiro_match_id].shape}")
    print(f"Vencedor (100 = Blue; 200 = Red): "
          f"{vencedores[primeiro_match_id]}")

Partidas carregadas: 100
Partidas organizadas: 100
Partidas descartadas na organização: 0
Características por equipe: 70

Partida de exemplo: BR1_3288893506
Formato da matriz: (23, 2, 70)
Vencedor (100 = Blue; 200 = Red): 100


In [6]:
# ============================================================
# CÉLULA 3 — INSPEÇÃO E VALIDAÇÃO DAS SÉRIES TEMPORAIS
# ============================================================

# ------------------------------------------------------------
# 1. Resumo geral
# ------------------------------------------------------------

print("RESUMO DAS SÉRIES TEMPORAIS")
print("=" * 60)

print(f"Partidas organizadas: {len(partidas)}")
print(f"Características por equipe: {len(colunas_features)}")

if not partidas:
    raise ValueError(
        "Nenhuma partida foi organizada. "
        "Verifique os dados e a célula 2."
    )

# ------------------------------------------------------------
# 2. Distribuição da quantidade de instantes por partida
# ------------------------------------------------------------

comprimentos = np.array([
    matriz.shape[0]
    for matriz in partidas.values()
])

print("\nQUANTIDADE DE INSTANTES POR PARTIDA")
print("=" * 60)

print(f"Mínimo:  {comprimentos.min()}")
print(f"Máximo:  {comprimentos.max()}")
print(f"Média:   {comprimentos.mean():.2f}")
print(f"Mediana: {np.median(comprimentos):.2f}")

print("\nDistribuição dos comprimentos:")
print(
    pd.Series(comprimentos)
    .describe()
    .round(2)
)

# ------------------------------------------------------------
# 3. Verificação dos formatos das matrizes
# ------------------------------------------------------------

formatos_invalidos = []

for match_id, matriz in partidas.items():
    if (
        matriz.ndim != 3
        or matriz.shape[1] != 2
        or matriz.shape[2] != len(colunas_features)
    ):
        formatos_invalidos.append(match_id)

print("\nVALIDAÇÃO DOS FORMATOS")
print("=" * 60)

print(f"Partidas com formato inválido: {len(formatos_invalidos)}")

# ------------------------------------------------------------
# 4. Verificação de valores ausentes e infinitos
# ------------------------------------------------------------

total_nan = 0
total_inf = 0
partidas_com_nan = 0
partidas_com_inf = 0

for matriz in partidas.values():
    mascara_nan = np.isnan(matriz)
    mascara_inf = np.isinf(matriz)

    quantidade_nan = mascara_nan.sum()
    quantidade_inf = mascara_inf.sum()

    total_nan += quantidade_nan
    total_inf += quantidade_inf

    if quantidade_nan > 0:
        partidas_com_nan += 1

    if quantidade_inf > 0:
        partidas_com_inf += 1

print("\nVALORES AUSENTES E INFINITOS")
print("=" * 60)

print(f"Valores NaN: {total_nan}")
print(f"Partidas com NaN: {partidas_com_nan}")
print(f"Valores infinitos: {total_inf}")
print(f"Partidas com infinitos: {partidas_com_inf}")

# ------------------------------------------------------------
# 5. Verificação dos alvos
# ------------------------------------------------------------

contagem_vencedores = pd.Series(
    list(vencedores.values())
).value_counts().reindex([100, 200], fill_value=0)

print("\nDISTRIBUIÇÃO DOS VENCEDORES")
print("=" * 60)

print(f"Blue (100): {contagem_vencedores.loc[100]}")
print(f"Red (200):  {contagem_vencedores.loc[200]}")

# ------------------------------------------------------------
# 6. Inspeção de uma partida
# ------------------------------------------------------------

match_id_exemplo = next(iter(partidas))
matriz_exemplo = partidas[match_id_exemplo]

print("\nEXEMPLO DE UMA PARTIDA")
print("=" * 60)

print(f"ID: {match_id_exemplo}")
print(f"Formato: {matriz_exemplo.shape}")
print(f"Vencedor: {vencedores[match_id_exemplo]}")

print("\nPrimeiro instante — Blue (100):")
display(
    pd.DataFrame(
        matriz_exemplo[0, 0, :].reshape(1, -1),
        columns=colunas_features
    )
)

print("\nPrimeiro instante — Red (200):")
display(
    pd.DataFrame(
        matriz_exemplo[0, 1, :].reshape(1, -1),
        columns=colunas_features
    )
)

# ------------------------------------------------------------
# 7. Resumo da validação
# ------------------------------------------------------------

print("\nRESUMO DA VALIDAÇÃO")
print("=" * 60)

if formatos_invalidos:
    print("ATENÇÃO: existem matrizes com formatos inválidos.")

if total_nan > 0:
    print("ATENÇÃO: existem valores ausentes que precisam ser tratados.")

if total_inf > 0:
    print("ATENÇÃO: existem valores infinitos que precisam ser investigados.")

if (
    not formatos_invalidos
    and total_nan == 0
    and total_inf == 0
):
    print("As matrizes passaram nas verificações estruturais básicas.")

RESUMO DAS SÉRIES TEMPORAIS
Partidas organizadas: 100
Características por equipe: 70

QUANTIDADE DE INSTANTES POR PARTIDA
Mínimo:  21
Máximo:  46
Média:   28.85
Mediana: 28.00

Distribuição dos comprimentos:
count    100.00
mean      28.85
std        5.37
min       21.00
25%       25.00
50%       28.00
75%       31.25
max       46.00
dtype: float64

VALIDAÇÃO DOS FORMATOS
Partidas com formato inválido: 0

VALORES AUSENTES E INFINITOS
Valores NaN: 0
Partidas com NaN: 0
Valores infinitos: 0
Partidas com infinitos: 0

DISTRIBUIÇÃO DOS VENCEDORES
Blue (100): 41
Red (200):  59

EXEMPLO DE UMA PARTIDA
ID: BR1_3288893506
Formato: (23, 2, 70)
Vencedor: 100

Primeiro instante — Blue (100):


,championStats.abilityHaste,championStats.abilityPower,championStats.armor,championStats.attackDamage,championStats.health,currentGold,damageStats.magicDamageDone,damageStats.magicDamageDoneToChampions,damageStats.magicDamageTaken,damageStats.physicalDamageDone,...,diff_kills,diff_deaths,diff_assists,diff_dragons,diff_elder_dragons,diff_barons,diff_rift_heralds,diff_atakhan,diff_towers_destroyed,diff_inhibitors_destroyed
0,0.0,0.0,162.0,303.0,3136.0,2100.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Primeiro instante — Red (200):


,championStats.abilityHaste,championStats.abilityPower,championStats.armor,championStats.attackDamage,championStats.health,currentGold,damageStats.magicDamageDone,damageStats.magicDamageDoneToChampions,damageStats.magicDamageTaken,damageStats.physicalDamageDone,...,diff_kills,diff_deaths,diff_assists,diff_dragons,diff_elder_dragons,diff_barons,diff_rift_heralds,diff_atakhan,diff_towers_destroyed,diff_inhibitors_destroyed
0,0.0,0.0,165.0,310.0,3252.0,2100.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



RESUMO DA VALIDAÇÃO
As matrizes passaram nas verificações estruturais básicas.
